In [ ]:
%load_ext watermark


In [ ]:
import itertools as it
import os

import matplotlib as mpl
from matplotlib import pyplot as plt
import numpy as np
import pandas as pd
from phyloframe import _auxlib as pfa
from phyloframe import legacy as pfl
from pyfonts import load_google_font
import seaborn as sns
from teeplot import teeplot as tp

import pylib


In [ ]:
%watermark -diwmuv -iv


In [ ]:
teeplot_subdir = os.environ.get("NOTEBOOK_NAME", "2026-03-12-btr-lineage")
teeplot_subdir


In [ ]:
pfa.seed_random(1)


In [ ]:
font = load_google_font("Merriweather", weight=300)
mpl.font_manager.fontManager.addfont(font.get_file())
plt.rcParams["font.family"] = font.get_name()


## Prep Data


In [ ]:
df_pure = pfl.alifestd_join_roots(
    pd.read_parquet("https://osf.io/mpkrd/download"),
)
df_pure


In [ ]:
df_pure = (
    df_pure
    .pipe(pfl.alifestd_to_working_format)
    .pipe(pfl.alifestd_collapse_unifurcations)
    .pipe(pfl.alifestd_to_working_format)
)
df_pure


In [ ]:
df_pure["x"] = df_pure["position"] // df_pure["nCol"]
df_pure["x_"] = df_pure["x"] / df_pure["nRow"]
df_pure["y"] = df_pure["position"] % df_pure["nCol"]
df_pure["y_"] = df_pure["y"] / df_pure["nCol"]

df_pure["origin_time"] = df_pure["dstream_rank"]


In [ ]:
df_pure_lineage = df_pure.copy()
df_pure_lineage["extant"] = df_pure["is_lineage"]
df_pure_lineage = pfl.alifestd_prune_extinct_lineages_asexual(df_pure_lineage)
df_pure_lineage


In [ ]:
df_pure_with_foliage = df_pure.copy()
df_pure_with_foliage["extant"] = (
    (df_pure["is_foliage"] & (np.random.rand(len(df_pure)) < 0.03))
    | df_pure["is_lineage"]
)
df_pure_with_foliage = pfl.alifestd_prune_extinct_lineages_asexual(df_pure_with_foliage)
df_pure_with_foliage


## Plot Lineage Tree


In [ ]:
for regime, seed in it.product(
    ("pure",),
    range(1),
):
    df = df_pure_lineage.copy()
    df["extant"] = df["origin_time"] > df["origin_time"].max() / 3
    df = (
        df.pipe(pfl.alifestd_prune_extinct_lineages_asexual)
        .pipe(pfl.alifestd_to_working_format)
        .pipe(
            pfl.alifestd_downsample_tips_lineage_asexual,
            criterion_target="layer",
            n_downsample=800,
            seed=seed,
        )
        .pipe(pfl.alifestd_collapse_unifurcations)
        .pipe(pfl.alifestd_delete_unifurcating_roots_asexual)
        .pipe(pfl.alifestd_mark_node_depth_asexual)
    )

    for layout in "vertical",:
        with tp.teed(
            pylib.chloropleth.draw_ctree,
            df,
            x="x_",
            y="y_",
            cmap=pylib.cmap.bcyr.get_color,
            layout=layout,
            scatter_kws=dict(
                edgecolor="none",
                s=4,
            ),
            scatter_shuffle=1,
            tree_kws=dict(
                edge=dict(
                    color="gainsboro",
                    linewidth=0.5,
                ),
                margins=-0.05,
            ),
            teeplot_outattrs={
                "regime": regime, "seed": seed
            },
            teeplot_subdir=teeplot_subdir,
        ) as teed:
            teed.invert_yaxis()
            teed.figure.set_size_inches(3, 1.33)

####
    df_ = df_pure_with_foliage.copy()
    df_["x__"] = df_["x_"].copy()
    df_["y__"] = df_["y_"].copy()

    mask = df_["dstream_data_id"].isin(df["dstream_data_id"])
    df_.loc[~mask, "x_"] = pd.NA
    df_.loc[~mask, "y_"] = pd.NA


    for layout in "vertical",:

        with tp.teed(
            pylib.chloropleth.draw_ctree,
            df_,
            x="x__",
            y="y__",
            cmap=lambda *args, **kwargs: sns.set_hls_values(
                pylib.cmap.bcyr.get_color(*args, **kwargs),
                l=0.95,
                s=0.2,
            ),
            layout=layout,
            scatter_kws=dict(
                edgecolor="none",
                s=4,
            ),
            scatter_shuffle=1,
            tree_kws=dict(
                edge=dict(
                    color="#f4f4f4",
                    linewidth=0.5,
                ),
                margins=-0.05,
            ),
            teeplot_outattrs={"regime": f"{regime}-mask", "seed": seed},
            teeplot_subdir=teeplot_subdir,
        ) as teed:
            pylib.chloropleth.draw_ctree(
                df_,
                x="x_",
                y="y_",
                cmap=pylib.cmap.bcyr.get_color,
                layout=layout,
                scatter_kws=dict(
                    edgecolor="none",
                    s=4,
                ),
                scatter_shuffle=1,
                tree_kws=dict(
                    edge=dict(
                        alpha=0.0,
                        color="#f4f4f4",
                        linewidth=0.5,
                    ),
                    margins=-0.05,
                ),
            )
            teed.invert_yaxis()
            teed.figure.set_size_inches(3, 1.33)
####

    data = df.copy().sort_values(by="node_depth").dropna(subset=["x_", "y_"])
    data = data[data["is_leaf"]].copy()
    data = data.reset_index(drop=True).copy()
    data.index = pd.to_timedelta(data["origin_time"], unit="s")
    data = data.resample("2000s").nearest().copy()
    with tp.teed(
        pylib.chloropleth.draw_cscatter,
        data,
        x="x",
        y="y",
        cmap=pylib.cmap.bcyr.get_color,
        despine=True,
        major=100,
        minor=None,
        xmax=1170,
        ymax=755,
        scatter_kws=dict(
            alpha=0.7,
            color="none",
            edgecolor="white",
            linewidth=1,
            linestyle="-",
            marker="o",
            s=10,
            zorder=120,
        ),
        teeplot_outattrs={
            "cmap": "bcyr",
            "regime": regime,
            "seed": seed,
        },
        teeplot_subdir=teeplot_subdir,
    ) as teed:
        for i in range(len(data)):
            teed.plot(
                data["x"][i:],
                data["y"][i:],
                alpha=1 / len(data),
                color="black",
                lw=6,
            )
        teed.plot(
            data["x"],
            data["y"],
            alpha=0.2,
            color="black",
            lw=6,
            zorder=150,
        )
        teed.set_xlabel(None)
        teed.set_ylabel(None)
        teed.set_aspect("equal")
        fig = teed.figure
        fig.set_size_inches(3, 2)
        fig.tight_layout()
